# NB05: scarHRD genomic labels (Marquard / Sztupinszki)

Adds the scarHRD continuous score and its binary label to the patient
labels table. The scarHRD score is taken from the TCGA DDR resource
bundle (`DDRscores.tsv` / `Scores.tsv` / `Samples.tsv`).

**Manuscript reference** (Methods, Supp Tables S3, S5):
- HRD = LOH + TAI + LST (Marquard et al. 2015 / Sztupinszki et al. via the
  scarHRD R package: https://github.com/sztup/scarHRD)
- Binary threshold: scarHRD >= 33 (top 20th percentile of training distribution)
- 8,109 patients have scarHRD scores out of 10,797 with WSIs
- HRD prevalence: 20.4% (n = 1,655)

**Required env**: `WORKSPACE`, `DDR_DIR` (TCGA DDR resource directory).
Input from NB04: `labels/labels.parquet`.
**Outputs**: updates `labels/labels.parquet` and `labels/labels.csv` with
`HRD_genomic`, `HRD`, and `HRD_top20` columns.

In [ ]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
DDR_DIR = Path(os.environ["DDR_DIR"]).resolve()
LABELS_DIR = WORKSPACE / "labels"
RESULTS_DIR = WORKSPACE / "results" / "labels"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked threshold (Methods + Supp S5)
HRD_THRESHOLD = 33
TOP_FRAC = 0.20

print(f"DDR_DIR    : {DDR_DIR}")
print(f"threshold  : scarHRD >= {HRD_THRESHOLD} (top {int(TOP_FRAC * 100)}%)")


In [ ]:
# load labels.parquet from NB04
labels_pq = LABELS_DIR / "labels.parquet"
assert labels_pq.exists(), f"missing {labels_pq}; run NB04 first"
labels = pd.read_parquet(labels_pq)
print(f"loaded labels: {len(labels):,} patients")
print(f"columns: {list(labels.columns)}")

# normalize patient column to 12-char TCGA barcode
def norm12(s):
    s = pd.Series(s, dtype=str).str.upper().str.strip()
    return s.str.replace(r"[^A-Z0-9-]", "", regex=True).str.slice(0, 12)

labels["patient"] = norm12(labels["patient"])
labels = labels.drop_duplicates(subset=["patient"]).set_index("patient")


In [ ]:
# load DDR resource files
ddr_scores = DDR_DIR / "DDRscores.tsv"
ddr_rows = DDR_DIR / "Scores.tsv"
ddr_cols = DDR_DIR / "Samples.tsv"
for f in [ddr_scores, ddr_rows, ddr_cols]:
    assert f.exists(), f"missing DDR file: {f}"

S = pd.read_csv(ddr_scores, sep="\t", header=None, dtype=str)
row_meta = pd.read_csv(ddr_rows, sep="\t", header=None, dtype=str)
col_meta = pd.read_csv(ddr_cols, sep="\t", header=None, dtype=str)

assert S.shape[0] == len(col_meta) and S.shape[1] == len(row_meta), (
    f"DDR shape mismatch: scores={S.shape}, samples={col_meta.shape}, names={row_meta.shape}"
)

S.columns = row_meta.iloc[:, 0].astype(str).tolist()
print(f"DDR scores matrix: {S.shape}")
print(f"available scores (first 10): {list(S.columns)[:10]}")


In [ ]:
# auto-detect TCGA barcode column in Samples.tsv
def looks_like_tcga(x):
    return bool(re.match(r"^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}", str(x).upper().strip()))

best_col, best_share = None, -1.0
for c in col_meta.columns:
    share = float(col_meta[c].astype(str).map(looks_like_tcga).mean())
    if share > best_share:
        best_share, best_col = share, c

assert best_share >= 0.5, f"no Samples.tsv column has >=50% TCGA barcodes; check input"
S.index = norm12(col_meta[best_col].astype(str))
print(f"barcode column: index {best_col} (TCGA-share={best_share:.1%})")
print(f"unique 12-char patients: {S.index.nunique():,}")


In [ ]:
# build HRD_genomic: prefer single 'HRD_Score' column if present, else sum LOH+TAI+LST
lut = {c.lower().replace("_", ""): c for c in S.columns}
def pick(name):
    return lut.get(name.lower().replace("_", ""))

col_hrd = pick("HRD_Score") or pick("HRDsum") or pick("HRD_Sum")
col_loh, col_tai, col_lst = pick("LOH"), pick("TAI"), pick("LST")

if col_hrd:
    hrd_numeric = pd.to_numeric(S[col_hrd], errors="coerce")
    hrd_source = "HRD_Score"
elif all([col_loh, col_tai, col_lst]):
    hrd_numeric = (
        pd.to_numeric(S[col_loh], errors="coerce")
        + pd.to_numeric(S[col_tai], errors="coerce")
        + pd.to_numeric(S[col_lst], errors="coerce")
    )
    hrd_source = "LOH+TAI+LST"
else:
    raise SystemExit(f"no HRD score in DDR file; columns: {list(S.columns)[:20]}")

geno = pd.DataFrame({"patient": S.index, "HRD_genomic": hrd_numeric})
geno = geno.dropna(subset=["patient"]).drop_duplicates(subset=["patient"]).set_index("patient")
print(f"HRD_genomic source : {hrd_source}")
print(f"HRD_genomic non-null: {geno['HRD_genomic'].notna().sum():,}")


In [ ]:
# merge into labels and apply manuscript threshold
labels["HRD_genomic"] = pd.to_numeric(geno["HRD_genomic"].reindex(labels.index), errors="coerce")
labels["HRD"] = labels["HRD_genomic"].astype(float)
labels["HRD_top20"] = (labels["HRD"] >= HRD_THRESHOLD).astype("Int64")

n_with_hrd = int(labels["HRD"].notna().sum())
n_pos = int((labels["HRD"] >= HRD_THRESHOLD).sum())
prev = 100.0 * n_pos / n_with_hrd if n_with_hrd else 0.0

print(f"patients with WSIs + scarHRD : {n_with_hrd:,}")
print(f"manuscript ref               : 8,109")
print(f"HRD-positive (>= {HRD_THRESHOLD})        : {n_pos:,} ({prev:.1f}%)")
print(f"manuscript ref               : 1,655 (20.4%)")


In [ ]:
# verify training-set top-20% threshold reproduces manuscript value of 33
train_hrd = labels.loc[labels["split"] == "train", "HRD"].dropna()
empirical_thr = float(np.quantile(train_hrd, 1.0 - TOP_FRAC)) if len(train_hrd) else float("nan")
match = abs(empirical_thr - HRD_THRESHOLD) < 1.0 if np.isfinite(empirical_thr) else False
print(f"empirical top-{int(TOP_FRAC*100)}% threshold on training: {empirical_thr:.2f}")
print(f"manuscript value: {HRD_THRESHOLD}")
print(f"match within 1.0: {match}")


In [ ]:
# write final labels (HRD column populated)
labels.reset_index(inplace=True)
out_pq = LABELS_DIR / "labels.parquet"
out_csv = LABELS_DIR / "labels.csv"
labels.to_parquet(out_pq, index=False)
labels.to_csv(out_csv, index=False)

diag = {
    "n_patients_total": int(len(labels)),
    "n_with_HRD": n_with_hrd,
    "n_HRD_positive": n_pos,
    "HRD_prevalence_pct": round(prev, 1),
    "threshold": HRD_THRESHOLD,
    "empirical_train_top20_thr": round(empirical_thr, 2) if np.isfinite(empirical_thr) else None,
    "hrd_source": hrd_source,
    "manuscript_refs": {
        "n_with_HRD": 8109,
        "n_HRD_positive": 1655,
        "HRD_prevalence_pct": 20.4,
        "threshold": 33,
    },
}
(RESULTS_DIR / "scarHRD_summary.json").write_text(json.dumps(diag, indent=2))
print(json.dumps(diag, indent=2))
print(f"\nwritten: {out_pq}")
